In [ ]:
# 1. Google Driveのマウント
from google.colab import drive
drive.mount('/content/drive')

# 2. データの読み込みパス設定（環境に合わせて変更してください）
# ※ マウントすると通常 'My Drive' になるため、パスを確認してください
DATA_DIR = "/content/drive/MyDrive/Kaggle"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
!cp /content/drive/MyDrive/Kaggle/agent_v2.py /content/agent.py
!wc -c /content/agent.py

62400 /content/agent.py


In [ ]:
import os

# 正しいzipファイルのパス
zip_path = os.path.join(DATA_DIR, 'simulator.zip')
# 展開先のディレクトリ（Colabのローカル環境）
extract_dir = '/content/simulator'

# 1. 展開前にzipの中身をリスト表示したい場合はこちら（コメントアウトを外してください）
# !unzip -l "{zip_path}"

# 2. zipファイルを展開
print(f"{zip_path} を {extract_dir} に展開します...")
!unzip -q "{zip_path}" -d "{extract_dir}"
print("展開が完了しました。\n")

# 3. 展開したディレクトリの中身を確認
print("--- 展開された内容 ---")
!ls -la "{extract_dir}"

/content/drive/MyDrive/Kaggle/simulator.zip を /content/simulator に展開します...
展開が完了しました。

--- 展開された内容 ---
total 12
drwxr-xr-x 3 root root 4096 Jul 26 02:16 .
drwxr-xr-x 1 root root 4096 Jul 26 02:16 ..
drwxrwxrwx 8 root root 4096 Jul  6 06:45 simulator


In [ ]:
%cd /content/simulator/simulator
!pip install pybullet
!PYTHONPATH=. python scripts/run_test.py --help

/content/simulator/simulator
  Using cached pybullet-3.2.7.tar.gz (80.5 MB)
  Preparing metadata (setup.py) ... done
  Created wheel for pybullet: filename=pybullet-3.2.7-cp312-cp312-linux_x86_64.whl size=99873152 sha256=46ce24f616c9d216ed8f7622072e174eeaca50aee3f1e35d9cbb79072ecbb4d0
  Stored in directory: /root/.cache/pip/wheels/72/95/1d/b336e5ee612ae9a019bfff4dc0bedd100ee6f0570db205fdf8
Successfully built pybullet
pybullet build time: Jul 26 2026 02:21:55
usage: run_test.py [-h] [--config-path CONFIG_PATH]
                   [--module-path MODULE_PATH] [--result-dir RESULT_DIR]
                   [--result-fname RESULT_FNAME] [--render-mode RENDER_MODE]
                   [--verbose VERBOSE]

options:
  -h, --help            show this help message and exit
  --config-path CONFIG_PATH
                        config path
  --module-path MODULE_PATH
                        agent module path
  --result-dir RESULT_DIR
                        result dir
  --result-fname RESULT_FNAME
     

In [19]:
!cp /content/drive/MyDrive/Kaggle/agent_v2.py /content/agent.py
!wc -c /content/agent.py
!python3 -c "import ast; ast.parse(open('/content/agent.py').read()); print('agent.py syntax OK')"

62400 /content/agent.py
agent.py syntax OK


In [21]:
import os, time
os.makedirs("/content/results", exist_ok=True)
t0 = time.time()
!cp /content/agent.py /content/simulator/simulator/agent.py && cd /content/simulator/simulator && PYTHONPATH=. python scripts/run_test.py --config-path configs/sample_config.json --module-path "" --result-dir /content/results --result-fname evaluation_results.json
print("elapsed seconds:", time.time() - t0)

pybullet build time: Jul 26 2026 02:21:55
argv[0]=
pybullet build time: Jul 26 2026 02:21:55
Perform Optimization!
transport item 39
place item 39
transport item 33
place item 33
transport item 5
place item 5
transport item 9
place item 9
transport item 11
place item 11
transport item 19
place item 19
transport item 20
place item 20
transport item 22
collision: 22 and 11 at (np.float32(0.0), np.float32(-0.45684934), np.float32(0.32999998)), distance 0.00744408183399828
collision: 22 and 19 at (np.float32(0.0), np.float32(-0.45684934), np.float32(0.32999998)), distance 0.007636373485254236
item 22 not packable. removed.
item 11 not inside (hit boundary plane), [ 4.32782448e-05 -6.38032727e-01 -1.53004328e+00 -1.28196727e+00
 -5.82840969e-01 -9.50582512e-01 -4.59417488e-01]
close env.
closed runner.
argv[0]=
pybullet build time: Jul 26 2026 02:21:55
transport item 1
place item 1
transport item 4
place item 4
transport item 11
place item 11
transport item 3
place item 3
transport item 12


## **手法を大きく転換する**
配置候補生成はヒューリスティック → 各候補を特徴量化 → LightGBM/NNでスコアリング → 安定性・到達可能性でマスク

### READMEからわかったことまとめ（全文版）

シミュレータの `README.md` 全体から、本環境の仕様とルールについて以下のことが分かりました。

**1. シミュレータの概要**
- PyBulletを用いた物理演算ベースの環境で、Gymnasium APIに準拠。
- 横空きコンテナに手荷物を最適に積載するエージェントの開発が目的。

**2. 評価指標（スコア）**
評価完了後に出力される JSON (`evaluation_results.json`) には以下の項目が含まれます。
- `fill_score`: 充填率のスコア
- `cog_score`: 重心位置のスコア
- `stability_score`: 動的シミュレーション（崩れにくさ）のスコア
- `placement_score`: 優先手荷物（Priority）の配置スコア
- `soft_item_score`: ソフト貨物の配置スコア
- `num_placed_items`: 荷物全体に対する、無事に積載できた荷物の割合
- `time_results`: `optimize` および `policy` メソッドの最大処理時間

**3. エラーとプロセス検証 (`status`)**
途中で失敗した場合、以下のいずれかのチェックに引っかかったことがわかります。
- `is_included`: コンテナの内包判定（はみ出していないか）
- `is_valid`: 搬入経路の干渉チェック（奥から手前へと順に置けているか）
- `is_placed_safe`: 配置直後の定着確認（置いた瞬間に大きくズレたり落ちたりしないか）

**4. コンペティション提出時の注意点（重要）**
- **汎用性の担保**: 配布されている `sample_config.json` はあくまでローカル確認用。実際の評価環境では、コンテナの数やサイズ、荷物の種類・数・順番などが**全く異なるテストケース**が使用されます。特定のサイズに依存しない汎用的なアルゴリズムが求められます。
- **実行時間**: `optimize` や `policy` の呼び出しには制限時間があるため、アルゴリズムは高速である必要があります。
- **通信制限**: 評価基盤ではインターネット通信が遮断されるため、外部APIなどは使用できません。
- **提出形式**: 作成したエージェントディレクトリ全体をZIP化して提出します。

In [ ]:
import os

file_path = '/content/simulator/simulator/src/ground_handling/env.py'

print(f"--- {file_path} の内包判定ロジックを確認 ---")
if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    # 'not included' や内包判定に関わりそうな箇所を検索して前後を表示
    for i, line in enumerate(lines):
        if "not included" in line or "def is_included" in line:
            start = max(0, i - 10)
            end = min(len(lines), i + 20)
            print(f"\n>>> 見つかった箇所: {line.strip()} (Line {i+1})")
            print("-" * 60)
            for j in range(start, end):
                print(f"{j+1:4d}: {lines[j].rstrip()}")
            print("-" * 60)
else:
    print("ファイルが見つかりません。")


--- /content/simulator/simulator/src/ground_handling/env.py の内包判定ロジックを確認 ---


In [ ]:
!grep -rnw '/content/simulator/simulator/src' -e 'not included' -e 'def is_included'

/content/simulator/simulator/src/ground_handling/validator.py:80:            print(f'not included. {dots.tolist()}')


In [ ]:
file_path = '/content/simulator/simulator/src/ground_handling/validator.py'

print(f"--- {file_path} の内包判定ロジックを確認 ---")
if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    # 'not included' がある行を中心に前後30行を表示
    for i, line in enumerate(lines):
        if "not included" in line:
            start = max(0, i - 30)
            end = min(len(lines), i + 30)
            print(f"\n>>> 見つかった箇所: {line.strip()} (Line {i+1})")
            print("-" * 80)
            for j in range(start, end):
                print(f"{j+1:4d}: {lines[j].rstrip()}")
            print("-" * 80)
            break
else:
    print("ファイルが見つかりません。")


--- /content/simulator/simulator/src/ground_handling/validator.py の内包判定ロジックを確認 ---

>>> 見つかった箇所: print(f'not included. {dots.tolist()}') (Line 80)
--------------------------------------------------------------------------------
  50:         orientation_max = len(action_config['orientations']) - 1
  51:         if not (0 <= action['container_idx'] <= container_idx_max):
  52:             valid = False
  53:             truncated = True
  54:             info = {'status': {f'has_valid_action_index (index in container_idx must be in the range of {0} to {container_idx_max})': False}}
  55:         if not (0 <= action['item_idx'] <= visible_pool_max):
  56:             valid = False
  57:             truncated = True
  58:             info = {'status': {f'has_valid_action_index (index in item_idx must be in the range of {0} to {visible_pool_max})': False}}
  59:         if not (0 <= action['orientation'] <= orientation_max):
  60:             valid = False
  61:             truncated = Tru

In [ ]:
import os

# 展開されたsimulatorフォルダのパス
simulator_dir = os.path.join(extract_dir, 'simulator')

# 中身を確認
print(f"--- {simulator_dir} の中身 ---")
!ls -la "{simulator_dir}"

--- /content/simulator/simulator の中身 ---
total 80
drwxrwxrwx 8 root root  4096 Jul  6 06:45 .
drwxr-xr-x 3 root root  4096 Jul 26 02:16 ..
drwxrwxrwx 3 root root  4096 Jun 18 06:52 agents
drwxrwxrwx 2 root root  4096 Jul  3 08:33 configs
-rw-rw-rw- 1 root root   162 Mar 17 07:30 docker-compose.yml
drwxrwxrwx 2 root root  4096 Jun 18 03:06 dockerfiles
drwxrwxrwx 2 root root  4096 May 13 03:07 figs
-rw-rw-rw- 1 root root 36998 Jul  6 06:17 README.md
-rw-rw-rw- 1 root root   915 Jun 18 06:51 sample_submit.zip
drwxrwxrwx 3 root root  4096 Jul  6 06:44 scripts
drwxrwxrwx 3 root root  4096 Jun 18 03:06 src


In [22]:
import os
import ast

simulator_src_dir = '/content/simulator/simulator/src'
output_file = '/content/simulator_guide.md'

class SimulatorVisitor(ast.NodeVisitor):
    def __init__(self):
        self.results = []
        self.current_class = None

    def visit_ClassDef(self, node):
        doc = ast.get_docstring(node)
        doc_str = doc.strip().replace('\n', ' ') if doc else "No docstring"
        self.results.append(f"### Class: `{node.name}`\n- **概要**: {doc_str}\n")
        self.current_class = node.name
        self.generic_visit(node)
        self.current_class = None

    def visit_FunctionDef(self, node):
        doc = ast.get_docstring(node)
        doc_str = doc.strip().replace('\n', ' ') if doc else "No docstring"
        args = [a.arg for a in node.args.args]

        # _で始まるプライベートメソッドは除外（必要なら外してください）
        if node.name.startswith('_') and node.name != '__init__':
            pass
        elif self.current_class:
            self.results.append(f"- **Method**: `{self.current_class}.{node.name}({', '.join(args)})`\n  - 説明: {doc_str}")
        else:
            self.results.append(f"### Function: `{node.name}({', '.join(args)})`\n- **説明**: {doc_str}\n")
        self.generic_visit(node)

markdown_lines = ["# シミュレーター解体書 (Simulator Functions Guide)\n"]
markdown_lines.append("このドキュメントはシミュレータ内の全クラス・メソッドを自動抽出したものです。\n")

for root, dirs, files in os.walk(simulator_src_dir):
    for file in files:
        if file.endswith('.py'):
            filepath = os.path.join(root, file)
            rel_path = os.path.relpath(filepath, simulator_src_dir)
            markdown_lines.append(f"## 📁 File: `{rel_path}`\n")

            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
            try:
                tree = ast.parse(content)
                visitor = SimulatorVisitor()
                visitor.visit(tree)

                if not visitor.results:
                    markdown_lines.append("*クラスや関数の定義はありません*\n")
                else:
                    markdown_lines.extend(visitor.results)
                markdown_lines.append("\n---\n")
            except Exception as e:
                markdown_lines.append(f"Error parsing file: {e}\n")

# ファイルに書き込み
with open(output_file, 'w', encoding='utf-8') as f:
    f.write('\n'.join(markdown_lines))

print(f"🎉 解体書を {output_file} に作成しました。\n先頭の数行をプレビュー表示します：\n")
print("="*60)
with open(output_file, 'r', encoding='utf-8') as f:
    print(''.join(f.readlines()[:50]))
print("="*60)
print(f"\n左のフォルダアイコンから {output_file} をダブルクリックするか、ダウンロードして全体を確認してください。")

🎉 解体書を /content/simulator_guide.md に作成しました。
先頭の数行をプレビュー表示します：

# シミュレーター解体書 (Simulator Functions Guide)

このドキュメントはシミュレータ内の全クラス・メソッドを自動抽出したものです。

## 📁 File: `ground_handling/evaluator.py`

### Class: `Evaluator`
- **概要**: コンテナ内の積載状態を計算する評価モジュール

- **Method**: `Evaluator.__init__(self, client, config)`
  - 説明: No docstring
- **Method**: `Evaluator.calculate_fill_rate(self, containers)`
  - 説明: 充填率の計算. コンテナ内に完全に収まっている荷物のみを抽出する
- **Method**: `Evaluator.evaluate(self, containers, total_items)`
  - 説明: 現在のコンテナの評価を辞書形式で返す

---

## 📁 File: `ground_handling/camera.py`

### Class: `Camera`
- **概要**: コンテナの状態を撮影し, ハイトマップ(高さ画像)を生成するクラス

- **Method**: `Camera.__init__(self, client, config)`
  - 説明: config:     target_pos (list): カメラが注視する中心座標 [x, y, z]     distance (float): 注視点からの距離     yaw, pitch, roll (float): カメラの回転角度（度）。                               pitch=-90で真下(トップダウン)を見る。                               横から見る場合は pitch=0 などに調整。     img_width, img_height (int): 出力画像の解像度     fov (float): 視野角     ne